<a href="https://colab.research.google.com/github/olyadiya/Intro_to_ML/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Гарри Дюбуа обучает CNN

Для выполнения задания я взяла отзывы со Steam на игру Disco Elysium (Гарри Дюбуа - главный персонаж игры).

Получилось спрарсить комментарии, так как у steam есть API /appreviews/ для выгрузки данных в json

In [25]:
import requests
import urllib.parse
import time
import json
import pandas as pd

In [23]:
def get_steam_reviews_paginated(app_id, target_count=500, batch_size=20):
    url = f"https://store.steampowered.com/appreviews/{app_id}"

    all_reviews = []
    cursor = "*"

    params = {"json": 1, "language": "english", "review_type": "all", "purchase_type": "all", "num_per_page": batch_size,"filter": "recent"}
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    while len(all_reviews) < target_count:
        params["cursor"] = cursor

        try:
            response = requests.get(url, params=params, headers=headers)
            response.raise_for_status()
            data = response.json()

            if data.get("success") != 1:
                print(f"Ошибка")
                break

            reviews_batch = data.get("reviews", [])
            if not reviews_batch:
                break

            all_reviews.extend(reviews_batch)
            cursor = data.get("cursor")
            if not cursor:
                break

            print(f"Загружено {len(reviews_batch)} отзывов. Всего: {len(all_reviews)}/{target_count}")

            time.sleep(0.5)

        except requests.exceptions.RequestException as e:
            break
        except json.JSONDecodeError as e:
            break
    return all_reviews

In [24]:
app_id = 632470
reviews = get_steam_reviews_paginated(app_id, target_count=500, batch_size=50)
if reviews:
    with open(f"steam_reviews_{app_id}.json", "w", encoding="utf-8") as f:
        json.dump(reviews, f, indent=2, ensure_ascii=False)

    for i, review in enumerate(reviews[:3], 1):
        text = review.get('review', '')[:150]
        print(f"\n{i}. [{ 'Рекомендует' if review.get('voted_up') else 'Не рекомендует' }] {text}...")

Загружено 50 отзывов. Всего: 50/500
Загружено 50 отзывов. Всего: 100/500
Загружено 50 отзывов. Всего: 150/500
Загружено 50 отзывов. Всего: 200/500
Загружено 50 отзывов. Всего: 250/500
Загружено 50 отзывов. Всего: 300/500
Загружено 50 отзывов. Всего: 350/500
Загружено 50 отзывов. Всего: 400/500
Загружено 50 отзывов. Всего: 450/500
Загружено 50 отзывов. Всего: 500/500

1. [Рекомендует] Man... after spending some time completing this game, I'd say Disco Elysium has PEAK storytelling. The story is really gripping you in. The characters...

2. [Не рекомендует] I got 3-4 hours in before i got the point or got hooked on this game, eventually it picks up the pace and things start getting interesting. I dont thi...

3. [Рекомендует] "Why did I fondle the s-h-i-t jacket again!" 10/10, absolute cinema!...


Далее работаем с полученными данными. Для обучения я преобразую их в бинарный датасет, то есть таблица данных будет иметь: id комментария, текст комментария и в столбце с оценкой 1, если комментарий положительный, и 0, если комментарий отрицательный.

In [53]:
with open('steam_reviews_632470.json', 'r', encoding='utf-8') as f:
    reviews = json.load(f)

In [54]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 10)

In [55]:
df = pd.DataFrame([
    {
        'review_id': r['recommendationid'],
        'rating': 1 if r['voted_up'] else 0,
        'review_text': r['review']
    }
    for r in reviews
])

In [56]:
print(df[['review_id', 'rating', 'review_text']].head(3).values)

[['227965508' 1
  "Man... after spending some time completing this game, I'd say Disco Elysium has PEAK storytelling. The story is really gripping you in. The characters are really interesting, especially my man Kim. I really like the the dialogue, world building, the art style, I could go on and on. It was a new experience for me, and it is amazing. 10/10 would replay it again.\n\nOne thing I don't really like is the running back and forth for a quest. It can be really slow, but it was worth it."]
 ['227960988' 0
  'I got 3-4 hours in before i got the point or got hooked on this game, eventually it picks up the pace and things start getting interesting. I dont think i ever liked the game, but i did like the story and world eventually. I think people who like table top rpgs and visual novels will like this game, but if you dont, the game is very slow and dialog gets annoying. If this was just a book i would have liked it better']
 ['227950483' 1
  '"Why did I fondle the s-h-i-t jacket 

In [57]:
df.to_csv('reviews_full.csv', index=False, encoding='utf-8')

In [58]:
print(df.head(10))
print(df['rating'].value_counts())

   review_id  rating  \
0  227965508       1   
1  227960988       0   
2  227950483       1   
3  227939575       1   
4  227939088       1   
5  227935026       1   
6  227933829       1   
7  227932269       1   
8  227925467       1   
9  227923289       0   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

Перехоим к работе с CNN

In [59]:
#на всякий случай удаляем незаполненные строки
df = df[['review_text', 'rating']].dropna()
df.head()

,review_text,rating
0,"Man... after spending some time completing this game, I'd say Disco Elysium has PEAK storytelling. The story is really gripping you in. The characters are really interesting, especially my man Kim. I really like the the dialogue, world building, the art style, I could go on and on. It was a new experience for me, and it is amazing. 10/10 would replay it again.\n\nOne thing I don't really like is the running back and forth for a quest. It can be really slow, but it was worth it.",1
1,"I got 3-4 hours in before i got the point or got hooked on this game, eventually it picks up the pace and things start getting interesting. I dont think i ever liked the game, but i did like the story and world eventually. I think people who like table top rpgs and visual novels will like this game, but if you dont, the game is very slow and dialog gets annoying. If this was just a book i would have liked it better",0
2,"""Why did I fondle the s-h-i-t jacket again!"" 10/10, absolute cinema!",1
3,good game,1
4,"The writing is the best writing in any video game, full stop, no argument. It is funny and sad and furious and tender, sometimes in the same sentence. I laughed out loud and then felt genuinely moved approximately fourteen times. The entire game is a dialogue system. There is no combat to speak of. You talk to people, you talk to objects, and most importantly you talk to the 24 competing skill voices living inside your detective's head, all of whom have opinions and most of whom are actively making things worse. Your Electrochemistry skill will encourage you to do drugs. Your Inland Empire will tell you the corpse has feelings. Your Encyclopedia will deliver an eight paragraph monologue about neckties at the worst possible moment. You cannot turn them off. They are you.",1


In [60]:
df.describe()

,rating
count,500.00000
mean,0.85400
std,0.35346
min,0.00000
25%,1.00000
50%,1.00000
75%,1.00000
max,1.00000


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_text  500 non-null    object
 1   rating       500 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 7.9+ KB


Создаем матрицы признаков

In [123]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['review_text']).toarray()

In [124]:
X[0]

array([0, 2, 0, ..., 0, 0, 0])

In [125]:
X.shape

(500, 3916)

In [126]:
#смотрим на срез
X[0][1500:1600]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [128]:
#векторизируем метки
y = df['rating'].values
print(y)

[1 0 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1
 1 1 0 0 0 1 0 0 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1
 1 1 1 0 1 1 1 1 0 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0
 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 0 1 1 1 1 1 1 0 1 1 1
 1 1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1
 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1
 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 0 1 0 1 1 1 1 1 0 1 0 1 1 1 1 1 0 0 1 1 0 1 0 1 1 1 1 1 0 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 0 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1
 1 0 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 0 1 0 1 1 0 1 1 0 1 1 0 1 1 1 1 1 1 1 1
 1 1 1 0 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1
 0 1 0 1 0 1 0 1 1 1 0 1 

создаю тестовую выборку

In [129]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (400, 3916)
Test: (100, 3916)


работаем с разрядностью

In [130]:
import torch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.int64).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

In [131]:
X_tensor[0], y_tensor[0]

(tensor([0., 2., 0.,  ..., 0., 0., 0.]), tensor(1))

создаём специальный объект для пакетной обработки

In [132]:
from torch.utils.data import DataLoader, TensorDataset

In [133]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

обучение на тестовой выборке (парметры как в коде в туториале)

In [134]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * (input_dim // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = CNN1D(X.shape[1], len(np.unique(y)))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.5959
Epoch 2, Loss: 0.4395
Epoch 3, Loss: 0.3697
Epoch 4, Loss: 0.3164
Epoch 5, Loss: 0.2876


In [135]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    _, predicted = torch.max(predictions, 1)

In [136]:
print(classification_report(y_test, predicted.numpy()))
print(confusion_matrix(y_test, predicted.numpy()))

              precision    recall  f1-score   support

           0       0.50      0.13      0.21        15
           1       0.86      0.98      0.92        85

    accuracy                           0.85       100
   macro avg       0.68      0.55      0.56       100
weighted avg       0.81      0.85      0.81       100

[[ 2 13]
 [ 2 83]]


Попробую поменять параметры (с copilot подобрала верные параметры, чтобы они не противоречили друг другу)

In [137]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [139]:
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=5, padding=2) #увеличила kernel size, padding и фильтры
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.fc1 = nn.Linear(128 * (input_dim // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = CNN1D(X.shape[1], len(np.unique(y)))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005) #learninf rate 0.001 -> 0.0005

for epoch in range(20): #увеличила количнство эпох
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.8013
Epoch 2, Loss: 0.4226
Epoch 3, Loss: 0.3997
Epoch 4, Loss: 0.3610
Epoch 5, Loss: 0.3032
Epoch 6, Loss: 0.2542
Epoch 7, Loss: 0.2046
Epoch 8, Loss: 0.1808
Epoch 9, Loss: 0.1493
Epoch 10, Loss: 0.1480
Epoch 11, Loss: 0.1298
Epoch 12, Loss: 0.1089
Epoch 13, Loss: 0.0799
Epoch 14, Loss: 0.0591
Epoch 15, Loss: 0.0483
Epoch 16, Loss: 0.0469
Epoch 17, Loss: 0.0380
Epoch 18, Loss: 0.0368
Epoch 19, Loss: 0.0294
Epoch 20, Loss: 0.0247


In [140]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    _, predicted = torch.max(predictions, 1)

In [141]:
print(classification_report(y_test, predicted.numpy()))
print(confusion_matrix(y_test, predicted.numpy()))

              precision    recall  f1-score   support

           0       0.50      0.33      0.40        15
           1       0.89      0.94      0.91        85

    accuracy                           0.85       100
   macro avg       0.69      0.64      0.66       100
weighted avg       0.83      0.85      0.84       100

[[ 5 10]
 [ 5 80]]


После увеличения числа фильтров с 32/64 до 64/128 и размера ядра с 3 до 5 качество модели немного снизилось. Уменьшение learning rate до 0.0005  сделало обучение менее стабильным.

переносим результаты обучения

In [142]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y, dtype=torch.int64).to(device)

model = CNN1D(X.shape[1], len(np.unique(y))).to(device)

In [143]:
torch.save(model.state_dict(), "cnn_reviews.pth")